In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from google.colab import drive
if os.path.exists('/content/inputs.json.gz'):
    D = '/content'
else:
    drive.mount('/content/drive')
    D = '/content/drive/MyDrive/temporal-ood-ddi'
os.makedirs(f'{D}/runs/preds', exist_ok=True)
print(D)

In [ ]:
!mkdir -p /content/w && tar xzf {D}/ddiood_code.tgz -C /content/w 2>/dev/null; cp {D}/inputs.json.gz /content/w/
!pip -q install rank_bm25 bitsandbytes accelerate

In [ ]:
JOBS = [
 "b2sentence-biomedbert-disjoint-s42",
 "b2mixed-biomedbert-disjoint-s42",
 "b1names-biomedbert-disjoint-s42"
]

In [ ]:
%cd /content/w
!python -m ddiood.gpu_jobs --bundle inputs.json.gz --out {D}/runs --only {' '.join(JOBS)} 2>&1 | tee -a {D}/runs/E3_drug_disjoint_split.log | grep --line-buffered -E '^(start|done|skip)|Error|Traceback'

In [ ]:
import os
done = sorted(j for j in JOBS if os.path.exists(f'{D}/runs/preds/{j}.json'))
print(f'{len(done)}/{len(JOBS)} jobs finished')
print('missing:', [j for j in JOBS if j not in done])